# Results Analysis — Ad-hoc vs SULO KG × GraphSAGE / R-GCN / TransE

Consolidates every experiment output listed in `CLAUDE.md`'s Experimental Goals
into the "Final results package" (section 6): three tables, three figures, plus
the supporting hyperparameter record and K-sweep sensitivity plots.

**Inputs this notebook reads** (all produced by other notebooks in this repo —
run them first if a section below reports files as missing):

| Source notebook | Produces |
|---|---|
| `extract_graph_geometry.ipynb` | `graph_geometry/rq1a_structural_comparison.csv`, `rq1a_subgraph_comparison.png`, `rq1a_summary_differences.png` |
| `RGCN.ipynb` | `rgcn/{base,sulo}_sequential_tuning_results.csv`, `..._final_unified_eval.pt`, `..._ablation_study_results.csv`, `..._entity_reuse_stats.csv`, `..._k_sweep_results.csv` |
| `graph_sage.ipynb` | same set under `graphsage/` |
| `transE.ipynb` | `transe/transe_results.csv`, `transe/{base,sulo}_sequential_tuning_results.csv` |

**This notebook must run on the same Databricks cluster as the others** — the
Table 2 section reconstructs and reloads the final RGCN/GraphSAGE checkpoints
with `torch`/`torch_geometric` to recover metrics that the tuning CSVs don't
themselves persist (overall F1, PR-AUC), so it needs `sys.path`, the mounted
`/Volumes/...` path, and a GPU/CPU environment exactly like the training
notebooks. It does not retrain anything — only re-runs evaluation on the
already-trained, already-saved final models.

All polished tables/figures this notebook produces are written to
`{volume}analysis/` alongside the raw per-model outputs.

In [ ]:
import os
import sys

repo_path = '/Workspace/Users/dversch@emea.corpdir.net/.bundle/KG_Model/prod/files'
if repo_path not in sys.path:
    sys.path.append(repo_path)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown

from torch_geometric.data import Data
from torch_geometric.nn import RGCNConv, SAGEConv

from node_prediction.gnn_tools import evaluate_test_set, parse_neighbors

In [10]:
volume = "/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/"

RGCN_DIR = f"{volume}rgcn/"
SAGE_DIR = f"{volume}graphsage/"
TRANSE_DIR = f"{volume}transe/"
GEOM_DIR = f"{volume}graph_geometry/"
LP_DIR = f"{volume}link_prediction/"
ANALYSIS_DIR = f"{volume}analysis/"
os.makedirs(ANALYSIS_DIR, exist_ok=True)

BASELINE_VARS = f"{volume}base_vars.pt"
SULO_VARS = f"{volume}sulo_vars.pt"

FEATURE_DIM = 1024
SCHEMAS = ['base', 'sulo']
SCHEMA_DISPLAY = {'base': 'Ad-hoc (Base)', 'sulo': 'SULO'}

In [11]:
# Fixed categorical color assignment (never re-cycled per chart) -- validated
# colorblind-safe with skill/dataviz's validate_palette.js for both group sizes
# used below (3-way model palette, 2-way KG palette).
MODEL_COLORS = {
    'RGCN': '#2a78d6',       # blue
    'GraphSAGE': '#eb6834',  # orange
    'TransE': '#1baf7a',     # aqua
}
KG_COLORS = {
    'Ad-hoc (Base)': '#1f77b4',  # matches extract_graph_geometry.ipynb's own Fig 1 palette
    'SULO': '#d62728',
}
MODELS_ORDER = list(MODEL_COLORS)
KGS_ORDER = list(KG_COLORS)

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': '#888888',
    'axes.grid': True,
    'grid.color': '#e6e6e6',
    'grid.linewidth': 0.6,
    'axes.axisbelow': True,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'legend.frameon': False,
})


def label_bars(ax, bars, fmt='{:.3f}', fontsize=8):
    """Direct value labels -- required 'relief' for any bar whose fill sits
    below 3:1 contrast against a white surface (e.g. the aqua model color)."""
    for b in bars:
        h = b.get_height()
        if h != h:  # NaN -- bar wasn't drawn, nothing to label
            continue
        ax.text(b.get_x() + b.get_width() / 2, h, fmt.format(h),
                 ha='center', va='bottom', fontsize=fontsize, color='#333333')


def save_fig(fig, name):
    path = f"{ANALYSIS_DIR}{name}"
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f"Saved figure: {path}")


def safe_read_csv(path, **kwargs):
    if not os.path.exists(path):
        print(f"[missing] {path}")
        return None
    return pd.read_csv(path, **kwargs)

In [12]:
# Quick "what's actually ready to plot yet" dashboard -- useful while tuning
# is still running on the cluster, since sections below degrade gracefully
# (skip + print a note) for anything not yet written.
def file_status(path):
    return "done" if os.path.exists(path) else "pending"


_status_rows = [
    ('Graph geometry: structural comparison (Table 1)', f"{GEOM_DIR}rq1a_structural_comparison.csv"),
    ('Graph geometry: subgraph figure (Figure 1a)', f"{GEOM_DIR}rq1a_subgraph_comparison.png"),
    ('Graph geometry: summary figure (Figure 1b)', f"{GEOM_DIR}rq1a_summary_differences.png"),
    ('RGCN base: tuning results', f"{RGCN_DIR}base_sequential_tuning_results.csv"),
    ('RGCN sulo: tuning results', f"{RGCN_DIR}sulo_sequential_tuning_results.csv"),
    ('RGCN base: final checkpoint', f"{RGCN_DIR}base_final_unified_eval.pt"),
    ('RGCN sulo: final checkpoint', f"{RGCN_DIR}sulo_final_unified_eval.pt"),
    ('RGCN base: ablation results', f"{RGCN_DIR}base_ablation_study_results.csv"),
    ('RGCN base: k-sweep results', f"{RGCN_DIR}base_k_sweep_results.csv"),
    ('GraphSAGE base: tuning results', f"{SAGE_DIR}base_sequential_tuning_results.csv"),
    ('GraphSAGE sulo: tuning results', f"{SAGE_DIR}sulo_sequential_tuning_results.csv"),
    ('GraphSAGE base: final checkpoint', f"{SAGE_DIR}base_final_unified_eval.pt"),
    ('GraphSAGE sulo: final checkpoint', f"{SAGE_DIR}sulo_final_unified_eval.pt"),
    ('GraphSAGE base: ablation results', f"{SAGE_DIR}base_ablation_study_results.csv"),
    ('GraphSAGE base: k-sweep results', f"{SAGE_DIR}base_k_sweep_results.csv"),
    ('TransE: results table', f"{TRANSE_DIR}transe_results.csv"),
    ('TransE base: tuning results', f"{TRANSE_DIR}base_sequential_tuning_results.csv"),
    ('TransE sulo: tuning results', f"{TRANSE_DIR}sulo_sequential_tuning_results.csv"),
    ('LP RGCN base: results', f"{LP_DIR}rgcn_base_lp_results.csv"),
    ('LP RGCN sulo: results', f"{LP_DIR}rgcn_sulo_lp_results.csv"),
    ('LP GraphSAGE base: results', f"{LP_DIR}graphsage_base_lp_results.csv"),
    ('LP GraphSAGE sulo: results', f"{LP_DIR}graphsage_sulo_lp_results.csv"),
    ('LP TransE base: results', f"{LP_DIR}transe_base_lp_results.csv"),
    ('LP TransE sulo: results', f"{LP_DIR}transe_sulo_lp_results.csv"),
]

display(pd.DataFrame(
    [{'artifact': label, 'status': file_status(path), 'path': path} for label, path in _status_rows]
))

,artifact,status,path
0,Graph geometry: structural comparison (Table 1),done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
1,Graph geometry: subgraph figure (Figure 1a),done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
2,Graph geometry: summary figure (Figure 1b),done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
3,RGCN base: tuning results,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
4,RGCN sulo: tuning results,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
5,RGCN base: final checkpoint,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
6,RGCN sulo: final checkpoint,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
7,RGCN base: ablation results,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
8,RGCN base: k-sweep results,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...
9,GraphSAGE base: tuning results,done,/Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-land...


## 1. KG Structural Comparison (RQ1a) — Table 1 & Figure 1

Both already computed by `extract_graph_geometry.ipynb`; this section loads,
polishes (adds a SULO-vs-Ad-hoc % change column), and re-displays them so they
sit alongside the rest of the results package.

In [13]:
comparison_df = safe_read_csv(f"{GEOM_DIR}rq1a_structural_comparison.csv", index_col='metric')

if comparison_df is not None:
    table1 = comparison_df.copy()

    def _pct_change(row):
        try:
            a, s = float(row.get('Ad-hoc (Base)')), float(row.get('SULO'))
        except (TypeError, ValueError):
            return np.nan
        return (s - a) / abs(a) * 100.0 if a != 0 else np.nan

    table1['SULO vs Ad-hoc (%)'] = table1.apply(_pct_change, axis=1)

    table1_path = f"{ANALYSIS_DIR}table1_kg_structural_comparison.csv"
    table1.to_csv(table1_path)
    print(f"Saved: {table1_path}")

    # ── Three-group split ──────────────────────────────────────────────
    # table1 above is kept exactly as before (one flat table, unchanged
    # file/column names, in case anything downstream reads it) -- this adds
    # a grouped VIEW on top of the same data, since a single ~35-row flat
    # table mixes three conceptually different kinds of measurement
    # together (schema design vs. populated graph shape vs. feature-to-Case
    # proximity), exactly the three groups CLAUDE.md's own experimental
    # design already describes separately.
    import re

    _schema_metrics = [m for m in [
        'num_classes_declared_local', 'num_subclass_relationships',
        'max_hierarchy_depth', 'avg_hierarchy_depth',
        'num_object_properties', 'num_datatype_properties',
        'num_generic_rdf_properties', 'relation_vocabulary_size_declared',
    ] if m in table1.index]

    _graph_metrics = [m for m in [
        'num_nodes', 'num_edges_triples', 'num_relation_types_observed',
        'avg_degree', 'median_degree', 'p99_degree_threshold',
        'num_high_degree_nodes', 'graph_density', 'num_connected_components',
        'largest_component_frac_nodes', 'num_intermediate_nodes',
        'intermediate_node_frac', 'avg_shortest_path_sampled',
        'shortest_path_sample_size', 'shortest_path_unreached_frac',
    ] if m in table1.index]

    _case_context_metrics = [m for m in ['num_cases', 'avg_case_neighbourhood_size']
                              if m in table1.index]

    # Feature-to-Case proximity properties are discovered dynamically from
    # whichever '{property}_mean_hops_to_case' / '{property}_pct_direct_1hop'
    # pairs actually exist in the data, rather than a hardcoded property
    # list -- so this keeps working if extract_graph_geometry.ipynb's own
    # tracked-property list ever grows or shrinks.
    _hop_props, _pct_props = set(), set()
    for _m in table1.index:
        _hm = re.match(r'^(.+)_mean_hops_to_case$', _m)
        if _hm:
            _hop_props.add(_hm.group(1))
        _pm = re.match(r'^(.+)_pct_direct_1hop$', _m)
        if _pm:
            _pct_props.add(_pm.group(1))
    _proximity_props = sorted(_hop_props | _pct_props)

    def _show_group(title, metrics):
        print(f"\n{'='*70}\n{title}\n{'='*70}")
        if not metrics:
            print("(no matching metrics found)")
            return
        display(table1.loc[metrics].style.format(precision=3, na_rep='—'))

    _show_group("Table 1a — Schema Design (ontology authoring, independent of instance data)",
                _schema_metrics)
    _show_group("Table 1b — Populated Instance Graph Shape", _graph_metrics)
    _show_group("Table 1c — Case-Neighborhood Context", _case_context_metrics)

    # Per-property feature-to-Case proximity, pivoted into one row per
    # property (instead of two flat, separately-ordered rows per property
    # in the original table) so the hop-distance / direct-1-hop pair for
    # each tracked property reads as a single comparable unit.
    print(f"\n{'='*70}\nTable 1d — Feature-to-Case Proximity, by tracked property\n{'='*70}")
    _prox_rows = []
    for _prop in _proximity_props:
        _hop_key = f"{_prop}_mean_hops_to_case"
        _pct_key = f"{_prop}_pct_direct_1hop"
        _row = {'Property': _prop}
        if _hop_key in table1.index:
            _row['Mean Hops (Base)'] = table1.loc[_hop_key, 'Ad-hoc (Base)']
            _row['Mean Hops (SULO)'] = table1.loc[_hop_key, 'SULO']
            _row['Hops Δ%'] = table1.loc[_hop_key, 'SULO vs Ad-hoc (%)']
        if _pct_key in table1.index:
            _row['%1-Hop (Base)'] = table1.loc[_pct_key, 'Ad-hoc (Base)']
            _row['%1-Hop (SULO)'] = table1.loc[_pct_key, 'SULO']
            _row['%1-Hop Δ%'] = table1.loc[_pct_key, 'SULO vs Ad-hoc (%)']
        _prox_rows.append(_row)

    if _prox_rows:
        table1d_proximity = pd.DataFrame(_prox_rows).set_index('Property')
        table1d_path = f"{ANALYSIS_DIR}table1d_feature_case_proximity.csv"
        table1d_proximity.to_csv(table1d_path)
        print(f"Saved: {table1d_path}")
        display(table1d_proximity.style.format(precision=3, na_rep='—'))
    else:
        table1d_path = None
        print("(no *_mean_hops_to_case / *_pct_direct_1hop metrics found)")

    # Safety net: surface anything not captured by the three groups above
    # instead of silently dropping it, in case rq1a_structural_comparison.csv
    # ever adds a metric this cell doesn't yet know how to categorize.
    _grouped = (set(_schema_metrics) | set(_graph_metrics) | set(_case_context_metrics)
                | {f"{p}_mean_hops_to_case" for p in _proximity_props}
                | {f"{p}_pct_direct_1hop" for p in _proximity_props})
    _ungrouped = [m for m in table1.index if m not in _grouped]
    if _ungrouped:
        print(f"\n[NOTE] {len(_ungrouped)} metric(s) not assigned to any group above "
              f"(shown here so nothing is silently dropped): {_ungrouped}")
        display(table1.loc[_ungrouped].style.format(precision=3, na_rep='—'))
else:
    print("Run extract_graph_geometry.ipynb first to produce Table 1 "
          "(rq1a_structural_comparison.csv).")


,Ad-hoc (Base),SULO,SULO vs Ad-hoc (%)
metric,,,
num_classes_declared_local,23.000,100.000,334.783
num_subclass_relationships,0.000,26.000,—
max_hierarchy_depth,0.000,2.000,—
avg_hierarchy_depth,0.000,0.310,—
num_object_properties,20.000,20.000,0.000
num_datatype_properties,18.000,18.000,0.000
num_generic_rdf_properties,38.000,0.000,-100.000
relation_vocabulary_size_declared,38.000,38.000,0.000


In [14]:
for _fname, _caption in [
    ('rq1a_subgraph_comparison.png', 'Figure 1a — 2-hop Case subgraph, Ad-hoc vs SULO'),
    ('rq1a_summary_differences.png', 'Figure 1b — Major structural differences summary'),
]:
    _path = f"{GEOM_DIR}{_fname}"
    if os.path.exists(_path):
        display(Markdown(f"**{_caption}**"))
        display(Image(filename=_path))
    else:
        print(f"[missing] {_path}")

<IPython.core.display.Markdown object>

## 2. Hyperparameter / Tuning Record (Experimental Goals §5)

Pulls the winning configuration out of each model's sequential-tuning CSV —
same selection rule each training notebook itself uses to hand off into its
Feature Ablation / K-sweep sections (peak validation ROC-AUC among Phase 3
verification rows, falling back to all completed runs if Phase 3 hasn't run
yet).

In [15]:
def best_gnn_config(tuning_csv_path):
    """
    Mirrors the winner-selection logic RGCN.ipynb/graph_sage.ipynb use in
    their own 'Retrieve metric & architecture winners' cell: peak
    val_roc_auc among Phase 3 verification rows, or all 'ok' rows if Phase 3
    hasn't completed yet. Returns (info_dict, full_df) or (None, df).
    """
    df = safe_read_csv(tuning_csv_path)
    if df is None or df.empty:
        return None, df

    ok = df[df['status'] == 'ok']
    phase3 = ok[ok['phase'].isin(['3_Verification', '3_Verification_Cached'])]
    if len(phase3) == 0:
        phase3 = ok
    if len(phase3) == 0:
        return None, df

    best = phase3.loc[phase3['val_roc_auc'].idxmax()]
    final = df[df['phase'] == '4_Final_Test_Evaluation']
    final_row = final.iloc[-1] if len(final) else None
    return {'best_val_row': best, 'final_test_row': final_row}, df


def best_transe_config(tuning_csv_path):
    """TransE analog of best_gnn_config -- its final phase is named
    '5_Final_Test_Evaluation' (transE.ipynb's run_transe_sequential_tuning),
    one phase later than RGCN/GraphSAGE since TransE inserts its own
    num_epochs sweep as Phase 3."""
    df = safe_read_csv(tuning_csv_path)
    if df is None or df.empty:
        return None, df

    ok = df[df['status'] == 'ok']
    if len(ok) == 0:
        return None, df

    best = ok.loc[ok['val_roc_auc'].idxmax()]
    final = df[df['phase'].astype(str).str.contains('Final', case=False, na=False)]
    final_row = final.iloc[-1] if len(final) else None
    return {'best_val_row': best, 'final_test_row': final_row}, df

In [16]:
hp_rows = []

for _model_name, _base_dir in [('RGCN', RGCN_DIR), ('GraphSAGE', SAGE_DIR)]:
    for _schema in SCHEMAS:
        _info, _ = best_gnn_config(f"{_base_dir}{_schema}_sequential_tuning_results.csv")
        if _info is None:
            hp_rows.append({'Model': _model_name, 'KG': SCHEMA_DISPLAY[_schema], 'Status': 'not available'})
            continue
        r = _info['best_val_row']
        hp_rows.append({
            'Model': _model_name, 'KG': SCHEMA_DISPLAY[_schema],
            'Hidden Dim': r.get('hidden_dim'), 'Layers': r.get('layers'),
            'Neighbors': r.get('neighbors'), 'Bases': r.get('num_bases'),
            'LR': r.get('lr'), 'Weight Decay': r.get('weight_decay'),
            'Best Epoch': r.get('best_epoch'), 'Val ROC-AUC': r.get('val_roc_auc'),
            'Status': 'ok',
        })

for _schema in SCHEMAS:
    _info, _ = best_transe_config(f"{TRANSE_DIR}{_schema}_sequential_tuning_results.csv")
    if _info is None:
        hp_rows.append({'Model': 'TransE', 'KG': SCHEMA_DISPLAY[_schema], 'Status': 'not available'})
        continue
    r = _info['best_val_row']
    hp_rows.append({
        'Model': 'TransE', 'KG': SCHEMA_DISPLAY[_schema],
        'Embedding Dim': r.get('emb_dim'), 'LR': r.get('lr'),
        'Epochs': r.get('num_epochs'), 'Val ROC-AUC': r.get('val_roc_auc'),
        'Status': 'ok',
    })

hp_table = pd.DataFrame(hp_rows)
_hp_col_order = ['Model', 'KG', 'Hidden Dim', 'Embedding Dim', 'Layers', 'Bases', 'Neighbors',
                  'LR', 'Weight Decay', 'Epochs', 'Best Epoch', 'Val ROC-AUC', 'Status']
hp_table = hp_table.reindex(columns=[c for c in _hp_col_order if c in hp_table.columns])

hp_table_path = f"{ANALYSIS_DIR}table_hyperparameters.csv"
hp_table.to_csv(hp_table_path, index=False)
display(hp_table.style.format(precision=5, na_rep='—'))
print(f"Saved: {hp_table_path}")

,Model,KG,Hidden Dim,Embedding Dim,Layers,Bases,Neighbors,LR,Weight Decay,Epochs,Best Epoch,Val ROC-AUC,Status
0,RGCN,Ad-hoc (Base),64.00000,—,2.00000,15.00000,"(25, 10)",0.00050,0.00010,—,4.00000,0.73211,ok
1,RGCN,SULO,256.00000,—,2.00000,45.00000,"(25, 10)",0.00100,0.00001,—,38.00000,0.71747,ok
2,GraphSAGE,Ad-hoc (Base),128.00000,—,2.00000,60.00000,"(25, 10)",0.00050,0.00050,—,7.00000,0.71914,ok
3,GraphSAGE,SULO,128.00000,—,3.00000,45.00000,"(15, 10, 5)",0.00010,0.00001,—,28.00000,0.71714,ok
4,TransE,Ad-hoc (Base),—,32.00000,—,—,—,0.05000,—,20.00000,—,0.51470,ok
5,TransE,SULO,—,32.00000,—,—,—,0.00100,—,30.00000,—,0.53630,ok


## 3. Model × KG Performance Comparison (RQ2a) — Table 2 & Figure 2

For RGCN/GraphSAGE, `run_sequential_optimization_with_verification`'s own
`4_Final_Test_Evaluation` CSV row now logs the full metric set directly
(`test_accuracy`/`test_precision`/`test_f1`/`test_pr_auc` alongside
`test_roc_auc` and the two class-specific F1s, via `gnn_tools.evaluate_final_test_row`).
CSVs produced before this fix (or by `gnn_tools.run_final_test_evaluation`'s
faster standalone re-run of just Phase 4 with already-tuned hyperparameters)
still won't have these columns populated for rows logged pre-fix. This
section reloads each model's saved `{schema}_final_unified_eval.pt`
checkpoint, rebuilds the exact winning architecture, and re-runs
`gnn_tools.evaluate_test_set` once on the untouched held-out test split as a
fallback to recover the complete metric set from those older rows without
retraining. This is a read-only evaluation pass, not a retrain.

TransE's `classify()`/`classify_logistic_regression()`/Phase-5 final-test
helpers in `transE.ipynb` now also compute an overall `f1` key (same
`f1_score(y_test, preds, zero_division=0)` call `gnn_tools.evaluate_test_set`
uses), flattened into `transe/transe_results.csv` as `classifier_f1` via
`build_result_row()`. Rows recorded before that fix won't have the column
populated and show up as `NaN` below -- re-run the relevant TransE cell(s)
to backfill them.

In [17]:
# Mirrored verbatim from RGCN.ipynb / graph_sage.ipynb (cell 9 in each) so
# `load_state_dict()` below lines up with the checkpoints those notebooks
# saved. If either notebook's model class changes, update it here too.

class RGCNModel(nn.Module):
    def __init__(self, in_channels, hidden_channels_list, num_relations,
                 vector_table, categorical_row_ids, num_bases=None,
                 out_channels=1, dropout=0.2):
        super().__init__()
        num_rows, emb_dim = vector_table.shape

        is_categorical = torch.zeros(num_rows, dtype=torch.bool)
        is_categorical[categorical_row_ids] = True
        self.register_buffer('is_categorical_mask', is_categorical)
        self.frozen_table = vector_table

        self.categorical_embedding = nn.Embedding(len(categorical_row_ids), emb_dim)
        with torch.no_grad():
            self.categorical_embedding.weight.copy_(
                vector_table[categorical_row_ids.to(vector_table.device)]
            )

        local_id = torch.full((num_rows,), -1, dtype=torch.long)
        local_id[categorical_row_ids] = torch.arange(len(categorical_row_ids))
        self.register_buffer('local_categorical_id', local_id)

        self.convs = nn.ModuleList()
        dims = [in_channels] + hidden_channels_list
        for i in range(len(dims) - 1):
            self.convs.append(RGCNConv(dims[i], dims[i + 1], num_relations=num_relations, num_bases=num_bases))
        self.convs.append(RGCNConv(dims[-1], out_channels, num_relations=num_relations, num_bases=num_bases))
        self.dropout = dropout

    def forward(self, node_row_ids, edge_index, edge_type, degree_feat):
        x = self.frozen_table[node_row_ids].clone()
        cat_mask = self.is_categorical_mask[node_row_ids]
        if cat_mask.any():
            local_ids = self.local_categorical_id[node_row_ids[cat_mask]]
            x[cat_mask] = self.categorical_embedding(local_ids)
        x = torch.cat([x, degree_feat.unsqueeze(1)], dim=1)
        for conv in self.convs[:-1]:
            x = conv(x, edge_index, edge_type)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index, edge_type)
        return x


class GraphSAGEModel(nn.Module):
    def __init__(self, in_channels, hidden_channels_list, num_relations, vector_table, categorical_row_ids,
                 num_bases=None, out_channels=1, aggr='mean', dropout=0.5):
        super().__init__()
        num_rows, emb_dim = vector_table.shape

        is_categorical = torch.zeros(num_rows, dtype=torch.bool)
        is_categorical[categorical_row_ids] = True
        self.register_buffer('is_categorical_mask', is_categorical)
        self.frozen_table = vector_table

        self.categorical_embedding = nn.Embedding(len(categorical_row_ids), emb_dim)
        with torch.no_grad():
            self.categorical_embedding.weight.copy_(
                vector_table[categorical_row_ids.to(vector_table.device)]
            )

        local_id = torch.full((num_rows,), -1, dtype=torch.long)
        local_id[categorical_row_ids] = torch.arange(len(categorical_row_ids))
        self.register_buffer('local_categorical_id', local_id)

        self.convs = nn.ModuleList()
        dims = [in_channels] + hidden_channels_list
        for i in range(len(dims) - 1):
            self.convs.append(SAGEConv(dims[i], dims[i + 1], aggr=aggr, project=True))
        self.convs.append(SAGEConv(dims[-1], out_channels, aggr=aggr, project=True))
        self.dropout = dropout

    def forward(self, node_row_ids, edge_index, edge_type, degree_feat):
        x = self.frozen_table[node_row_ids].clone()
        cat_mask = self.is_categorical_mask[node_row_ids]
        if cat_mask.any():
            local_ids = self.local_categorical_id[node_row_ids[cat_mask]]
            x[cat_mask] = self.categorical_embedding(local_ids)
        x = torch.cat([x, degree_feat.unsqueeze(1)], dim=1)
        for conv in self.convs[:-1]:
            x = conv(x, edge_index)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.convs[-1](x, edge_index)
        return x

In [18]:
def load_gnn_test_metrics(model_cls, schema_label, base_dir, vars_path, tuning_csv_path):
    """
    Reconstructs the winning architecture for (model_cls, schema_label) from
    its tuning CSV, loads the saved final_unified_eval.pt state dict, and
    re-runs gnn_tools.evaluate_test_set on the held-out test split -- this
    recovers the full metric set (overall F1, PR-AUC, accuracy, precision,
    recall) that the tuning CSV's own Phase 4 row doesn't persist.
    """
    info, tuning_df = best_gnn_config(tuning_csv_path)
    ckpt_path = f"{base_dir}{schema_label}_final_unified_eval.pt"

    if info is None:
        return {'status': 'no_tuning_csv'}
    if not os.path.exists(vars_path):
        return {'status': 'missing_vars_pt'}
    if not os.path.exists(ckpt_path):
        return {'status': 'missing_checkpoint'}

    best = info['best_val_row']

    saved = torch.load(vars_path, map_location='cpu', weights_only=False)
    graph_data = saved[f'{schema_label}_graph_data']
    test_indices = saved['x_test']
    vector_table = saved[f'{schema_label}_vector_table']

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    vector_table = vector_table.to(device)

    num_relations = graph_data.edge_type.max().item() + 1
    node_index = graph_data.n_id_to_vec_row
    usage_counts = torch.bincount(node_index, minlength=vector_table.shape[0])
    categorical_row_ids = torch.where(usage_counts > 1)[0]

    hidden_dim = int(best['hidden_dim'])
    layers = int(best['layers'])
    bases = best['num_bases']
    bases = None if pd.isna(bases) or bases == 'None' else int(float(bases))
    hidden_list = [hidden_dim] * (layers - 1)
    neighbors = parse_neighbors(best['neighbors'])

    model = model_cls(
        in_channels=FEATURE_DIM + 1,
        hidden_channels_list=hidden_list,
        num_relations=num_relations,
        vector_table=vector_table,
        categorical_row_ids=categorical_row_ids,
        num_bases=bases,
    )
    state = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(state['model_state_dict'])

    metrics = evaluate_test_set(model, graph_data, test_indices, batch_size=256, num_neighbors=neighbors)
    metrics.update({
        'status': 'ok',
        'hidden_dim': hidden_dim, 'layers': layers, 'bases': bases,
        'lr': best['lr'], 'weight_decay': best['weight_decay'], 'neighbors': str(neighbors),
        'best_epoch': best.get('best_epoch'),
        'runtime_sec': np.nan,
    })

    # The tuning row's own runtime_sec is for the winning *validation* run,
    # not the final combined-fit -- re-attach it here as the closest
    # available runtime figure for Table 2's "Runtime" column.
    same_arch = tuning_df[(tuning_df['status'] == 'ok') &
                           (tuning_df['hidden_dim'] == hidden_dim) &
                           (tuning_df['layers'] == layers)]
    if len(same_arch):
        metrics['runtime_sec'] = same_arch['runtime_sec'].iloc[-1]

    return metrics

In [19]:
table2_rows = []

for _model_name, _model_cls, _base_dir in [
    ('RGCN', RGCNModel, RGCN_DIR),
    ('GraphSAGE', GraphSAGEModel, SAGE_DIR),
]:
    for _schema in SCHEMAS:
        _vars_path = BASELINE_VARS if _schema == 'base' else SULO_VARS
        _tuning_csv = f"{_base_dir}{_schema}_sequential_tuning_results.csv"
        m = load_gnn_test_metrics(_model_cls, _schema, _base_dir, _vars_path, _tuning_csv)

        row = {'Model': _model_name, 'KG': SCHEMA_DISPLAY[_schema]}
        if m.get('status') != 'ok':
            row['Status'] = m.get('status', 'missing')
            table2_rows.append(row)
            continue

        row.update({
            'ROC-AUC': m['roc_auc'], 'F1': m['f1'], 'PR-AUC': m['pr_auc'],
            'Class 0 F1': m['class0_f1'], 'Class 1 F1': m['class1_f1'],
            'Accuracy': m['accuracy'], 'Precision': m['precision'], 'Recall': m['recall'],
            'Num Parameters': m['num_parameters'], 'Best Epoch': m['best_epoch'],
            'Runtime (s)': m['runtime_sec'], 'Status': 'ok',
        })
        table2_rows.append(row)

print(f"Collected {len([r for r in table2_rows if r.get('Status') == 'ok'])}/"
      f"{len(table2_rows)} RGCN/GraphSAGE rows from re-evaluated checkpoints.")

/databricks/python/lib/python3.12/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
/databricks/python/lib/python3.12/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
/databricks/python/lib/python3.12/site-packages/torch_geometric/loader/neighbor_loader.py:229: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(
/databricks/python/lib/python3.12/site-packages/torch_geometric/loader/neighbor_lo

In [ ]:
def load_transe_row(schema):
    df = safe_read_csv(f"{TRANSE_DIR}transe_results.csv")
    kg_label = SCHEMA_DISPLAY[schema]
    if df is None:
        return {'Model': 'TransE', 'KG': kg_label, 'Status': 'missing'}

    sub = df[df['kg'] == kg_label]
    if sub.empty:
        return {'Model': 'TransE', 'KG': kg_label, 'Status': 'missing'}

    # Prefer the tuned final model (transE.ipynb's Phase 5 result, recorded
    # as classifier='RandomForest (tuned)'); fall back to the untuned
    # fixed-hyperparameter run if tuning hasn't finished yet.
    preferred = sub[sub['classifier'] == 'RandomForest (tuned)']
    row = preferred.iloc[-1] if len(preferred) else sub.iloc[-1]

    return {
        'Model': 'TransE', 'KG': kg_label,
        'ROC-AUC': row.get('classifier_roc_auc'),
        'F1': row.get('classifier_f1'),  # NaN for older transe_results.csv rows recorded before transE.ipynb's classify()/classify_logistic_regression()/Phase 5 gained an 'f1' key
        'PR-AUC': row.get('classifier_pr_auc'),
        'Class 0 F1': row.get('classifier_class0_f1'), 'Class 1 F1': row.get('classifier_class1_f1'),
        'Accuracy': row.get('classifier_accuracy'), 'Precision': row.get('classifier_precision'),
        'Recall': row.get('classifier_recall'),
        'Num Parameters': row.get('embedding_num_parameters'),
        'Best Epoch': np.nan, 'Runtime (s)': row.get('embedding_runtime_seconds'),
        'Classifier': row.get('classifier'), 'Status': 'ok',
    }


for _schema in SCHEMAS:
    table2_rows.append(load_transe_row(_schema))

# XGBoost has no KG (tabular baseline) -- automl_tabular_baseline.ipynb's
# output columns already match table2_df's schema directly, one row.
_xgb_df = safe_read_csv(f"{AUTOML_DIR}tabular_baseline_results.csv")
if _xgb_df is not None:
    table2_rows.extend(_xgb_df.to_dict('records'))
else:
    table2_rows.append({'Model': 'XGBoost', 'KG': 'N/A (tabular baseline)', 'Status': 'missing'})

In [21]:
table2_df = pd.DataFrame(table2_rows)

_table2_col_order = ['Model', 'KG', 'ROC-AUC', 'F1', 'PR-AUC', 'Class 0 F1', 'Class 1 F1',
                      'Accuracy', 'Precision', 'Recall', 'Num Parameters', 'Best Epoch',
                      'Runtime (s)', 'Classifier', 'Status']
table2_df = table2_df.reindex(columns=[c for c in _table2_col_order if c in table2_df.columns])

table2_path = f"{ANALYSIS_DIR}table2_model_kg_performance.csv"
table2_df.to_csv(table2_path, index=False)
display(table2_df.style.format(precision=4, na_rep='—'))
print(f"Saved: {table2_path}")

# Indexed lookup for the figures below.
lookup = table2_df.set_index(['Model', 'KG'])

,Model,KG,ROC-AUC,F1,PR-AUC,Class 0 F1,Class 1 F1,Accuracy,Precision,Recall,Num Parameters,Best Epoch,Runtime (s),Classifier,Status
0,RGCN,Ad-hoc (Base),0.7145,0.7005,0.6903,0.5661,0.7005,0.6456,0.6094,0.8235,1075497,4.0000,0.0000,—,ok
1,RGCN,SULO,0.7038,0.6912,0.6984,0.5924,0.6912,0.6486,0.6336,0.7602,12112545,38.0000,0.0000,—,ok
2,GraphSAGE,Ad-hoc (Base),0.7040,0.6909,0.6804,0.5928,0.6909,0.6486,0.6198,0.7806,1353475,7.0000,0.0000,—,ok
3,GraphSAGE,SULO,0.6995,0.6763,0.6923,0.6025,0.6763,0.6432,0.6370,0.7208,1409027,28.0000,0.0000,—,ok
4,TransE,Ad-hoc (Base),0.4926,0.4777,0.5028,0.5025,0.4777,0.4904,0.4932,0.4630,43315904,—,411.9231,RandomForest (tuned),ok
5,TransE,SULO,0.5225,0.5099,0.5334,0.5227,0.5099,0.5164,0.5358,0.4865,480330816,—,37608.1233,RandomForest (tuned),ok


In [22]:
_metrics_to_plot = ['ROC-AUC', 'F1', 'PR-AUC', 'Class 0 F1', 'Class 1 F1']

fig, axes = plt.subplots(1, len(_metrics_to_plot), figsize=(4.2 * len(_metrics_to_plot), 4.2), sharey=True)
x = np.arange(len(MODELS_ORDER))
width = 0.35

for ax, metric in zip(axes, _metrics_to_plot):
    for i, kg in enumerate(KGS_ORDER):
        vals = []
        for m in MODELS_ORDER:
            try:
                v = lookup.loc[(m, kg), metric]
            except KeyError:
                v = np.nan
            vals.append(v if isinstance(v, (int, float, np.floating)) else np.nan)
        offset = (i - 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=kg, color=KG_COLORS[kg])
        label_bars(ax, bars, fmt='{:.3f}', fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(MODELS_ORDER, fontsize=9)
    ax.set_title(metric)
    ax.set_ylim(0, 1.05)

axes[0].set_ylabel('Score')
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=2)
fig.suptitle('Figure 2a — Model × KG performance (RQ2a)', y=1.18, fontsize=13, fontweight='bold')
plt.tight_layout()
save_fig(fig, 'figure2a_model_kg_performance.png')
plt.show()

Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure2a_model_kg_performance.png

In [23]:
# Slope chart: does SULO help or hurt each model, and is the direction
# consistent across architectures? Directly answers CLAUDE.md's "Ad-hoc ->
# SULO performance change for each model" + "identify whether the ontology
# effect is consistent across models" (RQ2a output requirements).
fig, ax = plt.subplots(figsize=(7, 5))
_metric = 'ROC-AUC'

for m in MODELS_ORDER:
    try:
        a = lookup.loc[(m, 'Ad-hoc (Base)'), _metric]
        s = lookup.loc[(m, 'SULO'), _metric]
    except KeyError:
        continue
    if pd.isna(a) or pd.isna(s):
        continue
    ax.plot([0, 1], [a, s], marker='o', color=MODEL_COLORS[m], linewidth=2, markersize=7)
    ax.text(1.04, s, f"{m}  ({s - a:+.3f})", va='center', fontsize=9, color=MODEL_COLORS[m])

ax.set_xticks([0, 1])
ax.set_xticklabels(KGS_ORDER)
ax.set_xlim(-0.15, 1.55)
ax.set_ylabel(_metric)
ax.set_title(f'Figure 2b — Ad-hoc → SULO {_metric} change by model')
plt.tight_layout()
save_fig(fig, 'figure2b_kg_delta_by_model.png')
plt.show()

Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure2b_kg_delta_by_model.png

## 4. Structure → Performance Relationship (RQ1b) — Figure 3

Joins Table 1's per-KG structural metrics against Table 2's per-(model, KG)
performance. Each structural metric only varies by KG (not by model), so
every panel below shows two x-positions (Ad-hoc / SULO) each carrying up to
three model points — read it as "does the KG-level structural difference
line up with a performance difference," not as a regression with real
statistical power (n=2 KGs).

The K-sweep panel below is the complementary RQ1b piece from
`gnn_tools.run_k_sweep` ("Experiment 2" in the module's own docstring):
holding every other hyperparameter fixed and varying only neighborhood depth
K, to see how sensitive each model/KG combination is to how much of the
graph's structure it's allowed to see.

In [24]:
if comparison_df is not None:
    _struct_metrics = [
        'num_relation_types_observed', 'relation_vocabulary_size_declared',
        'avg_degree', 'graph_density', 'avg_shortest_path_sampled',
        'max_hierarchy_depth', 'avg_case_neighbourhood_size',
        'CaseDescription_mean_hops_to_case', 'intermediate_node_frac',
    ]
    _struct_metrics = [m for m in _struct_metrics if m in comparison_df.index]

    _n = len(_struct_metrics)
    _ncols = 3
    _nrows = int(np.ceil(_n / _ncols)) if _n else 1
    fig, axes = plt.subplots(_nrows, _ncols, figsize=(5 * _ncols, 4 * _nrows))
    axes = np.atleast_1d(axes).flatten()

    _markers = {'RGCN': 'o', 'GraphSAGE': 's', 'TransE': '^'}

    for ax, sm in zip(axes, _struct_metrics):
        for m in MODELS_ORDER:
            for kg in KGS_ORDER:
                try:
                    xval = float(comparison_df.loc[sm, kg])
                    yval = lookup.loc[(m, kg), 'ROC-AUC']
                except (KeyError, ValueError, TypeError):
                    continue
                if pd.isna(xval) or pd.isna(yval):
                    continue
                ax.scatter(xval, yval, color=MODEL_COLORS[m], marker=_markers[m], s=80,
                           edgecolor='white', linewidth=0.6, zorder=3)
        ax.set_xlabel(sm, fontsize=8)
        ax.set_ylabel('ROC-AUC', fontsize=8)
        ax.set_title(sm, fontsize=9)
        ax.set_ylim(0, 1.05)

    for ax in axes[_n:]:
        ax.axis('off')

    _handles = [plt.Line2D([0], [0], marker=_markers[m], color='w', markerfacecolor=MODEL_COLORS[m],
                            markersize=9, label=m) for m in MODELS_ORDER]
    fig.legend(handles=_handles, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.04))
    fig.suptitle('Figure 3 — KG structural characteristics vs model performance (RQ1b)',
                 y=1.08, fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'figure3_structure_vs_performance.png')
    plt.show()
else:
    print("Table 1 (comparison_df) not available -- run extract_graph_geometry.ipynb first.")

Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure3_structure_vs_performance.png

In [25]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
_any_k_sweep = False
# Accumulates the SAME per-(model,schema) filtered/sorted frames already
# built for the plot below into one combined table -- previously this cell
# only ever produced figure3b, with no CSV of the underlying numbers behind
# it (every other figure in this notebook has a tableN_*.csv counterpart;
# this was the one exception). 'model'/'schema' are already real columns in
# each source {schema}_k_sweep_results.csv (gnn_tools.run_k_sweep writes
# them itself), so concatenating these frames directly is enough -- no
# manual tagging needed.
#
# non-'completed' rows (e.g. a genuine, batch-size-independent OOM -- Base's
# K=3 at its native 99th-percentile fan-out (47) induces a ~106K-node/seed
# neighborhood, ~130x SULO's K=3 at fan-out=9, which no batch-size retry can
# fix) are DELIBERATELY kept in table3b (just excluded from the line plot,
# which can't show a failed point) -- this is a reported finding (Base's
# higher native degree makes deep message passing computationally
# infeasible at true scale, a cost SULO's smaller neighborhoods don't
# incur), not a gap to hide by silently filtering it out.
_k_sweep_frames = []

for _model_name, _base_dir in [('RGCN', RGCN_DIR), ('GraphSAGE', SAGE_DIR)]:
    for _schema in SCHEMAS:
        df_raw = safe_read_csv(f"{_base_dir}{_schema}_k_sweep_results.csv")
        if df_raw is None or df_raw.empty:
            continue
        _k_sweep_frames.append(df_raw.sort_values('K'))
        _not_completed = df_raw[df_raw['status'] != 'completed']
        for _, _row in _not_completed.iterrows():
            print(f"[K-SWEEP] {_model_name}/{SCHEMA_DISPLAY[_schema]} K={_row['K']}: "
                  f"status='{_row['status']}' -- excluded from plot, kept in table3b as a finding.")
        df = df_raw[df_raw['status'] == 'completed'].sort_values('K')
        if df.empty:
            continue
        _any_k_sweep = True
        _style = '-' if _schema == 'base' else '--'
        _label = f"{_model_name} / {SCHEMA_DISPLAY[_schema]}"
        axes[0].plot(df['K'], df['ROC_AUC'], _style, marker='o', color=MODEL_COLORS[_model_name], label=_label)
        axes[1].plot(df['K'], df['F1'], _style, marker='o', color=MODEL_COLORS[_model_name], label=_label)

if _any_k_sweep:
    axes[0].set_title('ROC-AUC vs neighborhood depth (K)')
    axes[1].set_title('F1 vs neighborhood depth (K)')
    for ax in axes:
        ax.set_xlabel('K (hop depth)')
        ax.set_xticks([1, 2, 3])
    axes[0].set_ylabel('Score')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.15), ncol=2)
    plt.tight_layout()
    save_fig(fig, 'figure3b_k_sweep_sensitivity.png')
    plt.show()

    table3b_k_sweep = pd.concat(_k_sweep_frames, ignore_index=True)
    table3b_k_sweep_path = f"{ANALYSIS_DIR}table3b_k_sweep_results.csv"
    table3b_k_sweep.to_csv(table3b_k_sweep_path, index=False)
    print(f"Saved: {table3b_k_sweep_path} ({len(table3b_k_sweep)} rows)")
else:
    plt.close(fig)
    table3b_k_sweep_path = None
    print("No *_k_sweep_results.csv files found yet under rgcn/ or graphsage/.")

[K-SWEEP] RGCN/Ad-hoc (Base) K=3: status='oom' -- excluded from plot, kept in table3b as a finding.
Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure3b_k_sweep_sensitivity.png
Saved: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/table3b_k_sweep_results.csv (11 rows)

## 5. Feature / Relation Ablation (RQ2b) — Table 3

Ablation was deliberately only run for the single best-performing model
(per project decision — see the top-level request), not all three. This
section auto-detects whichever of `rgcn/`/`graphsage/` has
`*_ablation_study_results.csv` written and reports on that model; it does
not require both.

In [26]:
table3_df = pd.DataFrame()
table3_path = None
ablation_model, ablation_dir = None, None

for _name, _d in [('RGCN', RGCN_DIR), ('GraphSAGE', SAGE_DIR)]:
    if any(os.path.exists(f"{_d}{s}_ablation_study_results.csv") for s in SCHEMAS):
        ablation_model, ablation_dir = _name, _d
        break

if ablation_model is None:
    print("No *_ablation_study_results.csv found under rgcn/ or graphsage/ -- "
          "run the Feature Ablation section of the best model's notebook first.")
else:
    print(f"Best-model ablation results found for: {ablation_model}")
    _frames = []
    for _schema in SCHEMAS:
        df = safe_read_csv(f"{ablation_dir}{_schema}_ablation_study_results.csv")
        if df is not None:
            df.insert(0, 'kg', SCHEMA_DISPLAY[_schema])
            _frames.append(df)
    table3_df = pd.concat(_frames, ignore_index=True) if _frames else pd.DataFrame()

    table3_path = f"{ANALYSIS_DIR}table3_ablation_{ablation_model.lower()}.csv"
    table3_df.to_csv(table3_path, index=False)
    display(table3_df.style.format(precision=4, na_rep='—'))
    print(f"Saved: {table3_path}")

,kg,ablation,schema,edges_removed,pct_edges_removed,ROC_AUC,F1,PR_AUC,delta_ROC_AUC,RID,class0_F1,class1_F1,avg_neighborhood_size,high_signal_text_pct,fallback_noise_pct,runtime_sec,best_epoch,epochs_run,status
0,Ad-hoc (Base),Baseline Control,Baseline,0,0.0000,0.7201,0.6793,0.6977,0.0000,0.0000,0.6506,0.6793,13166.1000,48.2714,51.7286,1150.2784,4,20,completed
1,Ad-hoc (Base),Pruned Concept: CaseDescription,Baseline,117777,0.4299,0.7154,0.6527,0.6929,-0.0047,0.0108,0.6647,0.6527,13152.1000,47.9762,52.0238,1327.9467,7,23,completed
2,Ad-hoc (Base),Pruned Concept: Vehicle,Baseline,62914,0.2296,0.7110,0.6815,0.6897,-0.0090,0.0393,0.6313,0.6815,12986.6000,49.5279,50.4721,1305.6076,7,23,completed
3,Ad-hoc (Base),Pruned Concept: Account,Baseline,546566,1.9948,0.7183,0.6514,0.6909,-0.0018,0.0009,0.6609,0.6514,12994.7000,47.8864,52.1136,1186.4697,5,21,completed
4,Ad-hoc (Base),Pruned Concept: Task,Baseline,5965573,21.7725,0.7098,0.6662,0.6871,-0.0103,0.0005,0.6383,0.6662,11537.4000,42.3101,57.6899,1047.7058,4,20,completed
5,Ad-hoc (Base),Semantic Core (Paths Only),Baseline,23304317,85.0537,0.7283,0.6719,0.7075,0.0082,-0.0001,0.6776,0.6719,5761.2000,54.1607,45.8393,638.8534,3,19,completed
6,Ad-hoc (Base),Structural Core (Metadata Only),Baseline,8530228,31.1327,0.5892,0.6121,0.5648,-0.1309,0.0042,0.5330,0.6121,10150.2000,32.8359,67.1641,801.6197,1,17,completed
7,Ad-hoc (Base),Minimal Core,Baseline,25133095,91.7282,0.5720,0.6707,0.5597,-0.1480,0.0016,0.0199,0.6707,2426.4000,0.0000,100.0000,771.0738,17,33,completed
8,SULO,Baseline Control,SULO,0,0.0000,0.7002,0.6908,0.6994,0.0000,0.0000,0.5727,0.6908,21379.0000,5.4343,94.5657,1774.8634,32,48,completed
9,SULO,Pruned Concept: CaseDescription,SULO,248998,0.4357,0.6849,0.6781,0.6759,-0.0153,0.0352,0.5833,0.6781,21471.7000,4.9536,95.0464,1801.8365,32,48,completed


In [27]:
if ablation_model is not None and not table3_df.empty:
    for _schema in SCHEMAS:
        sub = table3_df[(table3_df['kg'] == SCHEMA_DISPLAY[_schema]) & (table3_df['status'] == 'completed')]
        sub = sub[sub['ablation'] != 'Baseline Control'].sort_values('delta_ROC_AUC')
        if sub.empty:
            continue

        fig, ax = plt.subplots(figsize=(8, max(3, 0.5 * len(sub))))
        colors = ['#e34948' if v < 0 else '#008300' for v in sub['delta_ROC_AUC']]
        bars = ax.barh(sub['ablation'], sub['delta_ROC_AUC'], color=colors)
        ax.axvline(0, color='#888888', linewidth=0.8)
        for b, v in zip(bars, sub['delta_ROC_AUC']):
            ax.text(v, b.get_y() + b.get_height() / 2, f'{v:+.3f}',
                    va='center', ha='left' if v >= 0 else 'right', fontsize=8)
        ax.set_xlabel('Δ ROC-AUC vs Baseline Control')
        ax.set_title(f'{ablation_model} feature-ablation impact — {SCHEMA_DISPLAY[_schema]}')
        plt.tight_layout()
        save_fig(fig, f'ablation_delta_{ablation_model.lower()}_{_schema}.png')
        plt.show()

Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/ablation_delta_rgcn_base.png
Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/ablation_delta_rgcn_sulo.png

In [28]:
if ablation_model is not None:
    for _schema in SCHEMAS:
        df = safe_read_csv(f"{ablation_dir}{_schema}_entity_reuse_stats.csv")
        if df is None or df.empty:
            continue

        fig, ax = plt.subplots(figsize=(7, 4))
        bars = ax.bar(df['entity'], df['reuse_ratio'], color=MODEL_COLORS.get(ablation_model, '#2a78d6'))
        label_bars(ax, bars, fmt='{:.2f}')
        ax.set_ylabel('Reuse ratio (instances / cases connected)')
        ax.set_title(f'Entity reuse — {ablation_model} / {SCHEMA_DISPLAY[_schema]}')
        ax.tick_params(axis='x', rotation=30)
        plt.tight_layout()
        save_fig(fig, f'entity_reuse_{ablation_model.lower()}_{_schema}.png')
        plt.show()

[missing] /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/rgcn/base_entity_reuse_stats.csv
[missing] /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/rgcn/sulo_entity_reuse_stats.csv

## 6. Link Prediction Comparison (Supplementary) — Table 4 & Figure 4

A separate diagnostic study, not part of `CLAUDE.md`'s four numbered
experiments: compares how well RGCN/GraphSAGE/TransE predict the missing
entity at one end of a triple, evaluated on the curated Vehicle/Account/Task
relations only (run via `link_prediction_rgcn.ipynb`/
`link_prediction_graphsage.ipynb`/`link_prediction_transe.ipynb`, reading
from `{volume}link_prediction/`).

All three models are scored by the same filtered, type-constrained ranking
function (`link_prediction_tools.evaluate_link_prediction`) -- same
candidate pools, held-out test triples, and MRR/Hits@k/AUC/AP/F1
definitions -- so results are directly comparable model-to-model.

Column-naming note: RGCN/GraphSAGE record the swept dimension as
`hidden_dim`, TransE as `emb_dim` (reconciled into one `Dim` column
below). `num_train_triples` means a different slice per model (RGCN/
GraphSAGE: the disjoint supervised slice; TransE: the full train split) --
don't compare that column across models.

Figure 4b (per-relation breakdown) is the one to read -- pooling is
dominated by whichever relation has the largest candidate pool.

In [29]:
def _lp_dim_value(row_or_df):
    """RGCN/GraphSAGE's *_lp_results.csv/*_lp_tuning_results.csv record the
    swept dimension as 'hidden_dim'; TransE's record it as 'emb_dim' -- see
    this section's intro cell for why. Returns whichever is present."""
    if 'hidden_dim' in row_or_df:
        return row_or_df['hidden_dim']
    return row_or_df.get('emb_dim')


def load_lp_results(model_name, schema, lp_dir):
    """Loads {model}_{schema}_lp_results.csv -- one row per relation plus a
    'pooled' aggregate row, all produced by the same
    link_prediction_tools.evaluate_link_prediction() call regardless of
    model. Returns None if the file doesn't exist yet."""
    df = safe_read_csv(f"{lp_dir}{model_name.lower()}_{schema}_lp_results.csv")
    if df is None:
        return None
    df = df.copy()
    df.insert(0, 'Model', model_name)
    df.insert(1, 'KG', SCHEMA_DISPLAY[schema])
    df['Dim'] = _lp_dim_value(df)
    return df


def load_lp_tuning(model_name, schema, lp_dir):
    """
    Loads {model}_{schema}_lp_tuning_results.csv and returns (best_row, df)
    for the winning dim (highest val_mrr among 'search' phase rows), or
    (None, df) if unavailable.

    Column set differs slightly by model, both reflected in what's returned
    here (missing keys just come back as None via Series.get()):
    RGCN/GraphSAGE (link_prediction_tools.run_small_lp_validation_search)
    also append a 'final_test' phase row with test_mrr_pooled, and don't log
    val_mrr_macro/best_epoch per search row. TransE's own search loop
    (link_prediction_transe.ipynb) logs val_mrr_macro/best_epoch per row but
    has no separate 'final_test' row -- its final test numbers live only in
    *_lp_results.csv's 'pooled' row.
    """
    df = safe_read_csv(f"{lp_dir}{model_name.lower()}_{schema}_lp_tuning_results.csv")
    if df is None or df.empty:
        return None, df
    search = df[df['phase'] == 'search']
    if search.empty:
        return None, df
    return search.loc[search['val_mrr'].idxmax()], df

In [30]:
_lp_frames = []
for _model_name in MODELS_ORDER:
    for _schema in SCHEMAS:
        _df = load_lp_results(_model_name, _schema, LP_DIR)
        if _df is not None:
            _lp_frames.append(_df)

table4_lp = pd.concat(_lp_frames, ignore_index=True) if _lp_frames else pd.DataFrame()

_table4_col_order = ['Model', 'KG', 'Dim', 'relation', 'mrr', 'hits@1', 'hits@3', 'hits@10',
                      'num_test_triples', 'candidate_pool_size', 'tie_rate', 'auc', 'ap', 'f1',
                      'num_train_triples', 'num_val_triples', 'status']
if not table4_lp.empty:
    table4_lp = table4_lp.reindex(columns=[c for c in _table4_col_order if c in table4_lp.columns])

table4_lp_path = f"{ANALYSIS_DIR}table4_lp_per_relation.csv"
table4_lp.to_csv(table4_lp_path, index=False)
if not table4_lp.empty:
    display(table4_lp.style.format(precision=4, na_rep='—'))
    print(f"Saved: {table4_lp_path}")
else:
    print("No *_lp_results.csv files found yet under link_prediction/ -- "
          "run link_prediction_{rgcn,graphsage,transe}.ipynb first.")

,Model,KG,Dim,relation,mrr,hits@1,hits@3,hits@10,num_test_triples,candidate_pool_size,tie_rate,auc,ap,f1,num_train_triples,num_val_triples,status
0,RGCN,Ad-hoc (Base),32,Vehicle,0.0701,0.0123,0.0453,0.1495,816,107.0000,0.0000,0.6797,0.6138,0.6250,1959.0000,816.0000,ok
1,RGCN,Ad-hoc (Base),32,Account,0.0018,0.0009,0.0009,0.0009,1098,10290.0000,0.0000,0.7611,0.6690,0.6840,2635.0000,1098.0000,ok
2,RGCN,Ad-hoc (Base),32,Task,0.0007,0.0001,0.0002,0.0007,11865,118658.0000,0.0228,0.9168,0.8807,0.8421,8000.0000,11865.0000,ok
3,RGCN,Ad-hoc (Base),32,pooled,0.0049,0.0009,0.0029,0.0095,13779,—,—,—,—,—,—,—,—
4,RGCN,SULO,32,Vehicle,0.0964,0.0375,0.0688,0.1875,160,107.0000,0.0187,0.6898,0.6500,0.6375,384.0000,160.0000,ok
5,RGCN,SULO,32,Account,0.0004,0.0000,0.0000,0.0000,1035,10264.0000,0.9990,0.7740,0.6942,0.6866,2484.0000,1035.0000,ok
6,RGCN,SULO,32,Task,0.0002,0.0001,0.0001,0.0005,10713,107137.0000,0.9985,0.8607,0.7813,0.7824,8000.0000,10713.0000,ok
7,RGCN,SULO,32,pooled,0.0015,0.0006,0.0010,0.0029,11908,—,—,—,—,—,—,—,—
8,GraphSAGE,Ad-hoc (Base),32,Vehicle,0.0640,0.0123,0.0355,0.1262,816,107.0000,0.0000,0.6768,0.6102,0.6164,1959.0000,816.0000,ok
9,GraphSAGE,Ad-hoc (Base),32,Account,0.0021,0.0000,0.0009,0.0027,1098,10290.0000,0.0064,0.7736,0.6854,0.6985,2635.0000,1098.0000,ok


In [31]:
table4_lp_pooled = (table4_lp[table4_lp['relation'] == 'pooled']
                     .drop(columns=['relation']).reset_index(drop=True)
                     if not table4_lp.empty else pd.DataFrame())

table4_lp_pooled_path = f"{ANALYSIS_DIR}table4b_lp_pooled_summary.csv"
table4_lp_pooled.to_csv(table4_lp_pooled_path, index=False)
if not table4_lp_pooled.empty:
    display(table4_lp_pooled.style.format(precision=4, na_rep='—'))
    print(f"Saved: {table4_lp_pooled_path}")

# Indexed lookup for Figure 4a below.
lp_lookup = table4_lp_pooled.set_index(['Model', 'KG']) if not table4_lp_pooled.empty else pd.DataFrame()

,Model,KG,Dim,mrr,hits@1,hits@3,hits@10,num_test_triples,candidate_pool_size,tie_rate,auc,ap,f1,num_train_triples,num_val_triples,status
0,RGCN,Ad-hoc (Base),32,0.0049,0.0009,0.0029,0.0095,13779,—,—,—,—,—,—,—,—
1,RGCN,SULO,32,0.0015,0.0006,0.0010,0.0029,11908,—,—,—,—,—,—,—,—
2,GraphSAGE,Ad-hoc (Base),32,0.0044,0.0007,0.0023,0.0083,13779,—,—,—,—,—,—,—,—
3,GraphSAGE,SULO,64,0.0015,0.0003,0.0009,0.0027,11908,—,—,—,—,—,—,—,—
4,TransE,Ad-hoc (Base),32,0.0045,0.0014,0.0029,0.0087,13779,—,—,—,—,—,—,—,—
5,TransE,SULO,32,0.0010,0.0002,0.0006,0.0022,11908,—,—,—,—,—,—,—,—


In [32]:
_lp_metrics_to_plot = ['mrr', 'hits@1', 'hits@3', 'hits@10']
_lp_metric_labels = {'mrr': 'MRR', 'hits@1': 'Hits@1', 'hits@3': 'Hits@3', 'hits@10': 'Hits@10'}

if not lp_lookup.empty:
    fig, axes = plt.subplots(1, len(_lp_metrics_to_plot), figsize=(4.2 * len(_lp_metrics_to_plot), 4.2), sharey=True)
    x = np.arange(len(MODELS_ORDER))
    width = 0.35

    for ax, metric in zip(axes, _lp_metrics_to_plot):
        for i, kg in enumerate(KGS_ORDER):
            vals = []
            for m in MODELS_ORDER:
                try:
                    v = lp_lookup.loc[(m, kg), metric]
                except KeyError:
                    v = np.nan
                vals.append(v if isinstance(v, (int, float, np.floating)) else np.nan)
            offset = (i - 0.5) * width
            bars = ax.bar(x + offset, vals, width, label=kg, color=KG_COLORS[kg])
            label_bars(ax, bars, fmt='{:.4f}', fontsize=7)
        ax.set_xticks(x)
        ax.set_xticklabels(MODELS_ORDER, fontsize=9)
        ax.set_title(_lp_metric_labels[metric])

    axes[0].set_ylabel('Score (pooled)')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=2)
    fig.suptitle('Figure 4a — Link prediction: pooled MRR/Hits@k by Model × KG',
                 y=1.18, fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'figure4a_lp_pooled_performance.png')
    plt.show()
else:
    print("No LP results loaded yet -- see Table 4 above.")

Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure4a_lp_pooled_performance.png

In [33]:
if not table4_lp.empty:
    _lp_relations = sorted(table4_lp.loc[table4_lp['relation'] != 'pooled', 'relation'].unique())

    fig, axes = plt.subplots(1, len(SCHEMAS), figsize=(7 * len(SCHEMAS), 4.5), sharey=True)
    axes = np.atleast_1d(axes)

    for ax, schema in zip(axes, SCHEMAS):
        kg = SCHEMA_DISPLAY[schema]
        x = np.arange(len(_lp_relations))
        width = 0.8 / len(MODELS_ORDER)
        for i, m in enumerate(MODELS_ORDER):
            vals = []
            for rel in _lp_relations:
                sub = table4_lp[(table4_lp['Model'] == m) & (table4_lp['KG'] == kg) & (table4_lp['relation'] == rel)]
                vals.append(sub['mrr'].iloc[0] if len(sub) else np.nan)
            offset = (i - (len(MODELS_ORDER) - 1) / 2) * width
            ax.bar(x + offset, vals, width, label=m, color=MODEL_COLORS[m])
        ax.set_xticks(x)
        ax.set_xticklabels(_lp_relations, rotation=30, ha='right', fontsize=8)
        ax.set_title(kg)

    axes[0].set_ylabel('MRR (per relation)')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=3)
    fig.suptitle('Figure 4b — Link prediction: per-relation MRR by Model, faceted by KG\n'
                 "(preferred over Figure 4a's pooled view -- candidate-pool size varies by\n"
                 "orders of magnitude across relations, see evaluate_link_prediction()'s docstring)",
                 y=1.22, fontsize=11, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'figure4b_lp_per_relation_mrr.png')
    plt.show()
else:
    print("No LP results loaded yet -- see Table 4 above.")

Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure4b_lp_per_relation_mrr.png

In [34]:
lp_hp_rows = []
for _model_name in MODELS_ORDER:
    for _schema in SCHEMAS:
        _best, _ = load_lp_tuning(_model_name, _schema, LP_DIR)
        if _best is None:
            lp_hp_rows.append({'Model': _model_name, 'KG': SCHEMA_DISPLAY[_schema], 'Status': 'not available'})
            continue
        lp_hp_rows.append({
            'Model': _model_name, 'KG': SCHEMA_DISPLAY[_schema],
            'Dim': _lp_dim_value(_best), 'Val MRR': _best.get('val_mrr'),
            'Val MRR (macro)': _best.get('val_mrr_macro'),
            'Best Epoch': _best.get('best_epoch'),
            'Runtime (s)': _best.get('runtime_sec'), 'Status': 'ok',
        })

lp_hp_table = pd.DataFrame(lp_hp_rows)
lp_hp_table_path = f"{ANALYSIS_DIR}table4c_lp_hyperparameters.csv"
lp_hp_table.to_csv(lp_hp_table_path, index=False)
display(lp_hp_table.style.format(precision=5, na_rep='—'))
print(f"Saved: {lp_hp_table_path}")

,Model,KG,Status
0,RGCN,Ad-hoc (Base),not available
1,RGCN,SULO,not available
2,GraphSAGE,Ad-hoc (Base),not available
3,GraphSAGE,SULO,not available
4,TransE,Ad-hoc (Base),not available
5,TransE,SULO,not available


In [35]:
fig, axes = plt.subplots(1, len(SCHEMAS), figsize=(7 * len(SCHEMAS), 4.5), sharey=True)
axes = np.atleast_1d(axes)
_any_lp_history = False

for ax, schema in zip(axes, SCHEMAS):
    kg = SCHEMA_DISPLAY[schema]
    for m in MODELS_ORDER:
        df = safe_read_csv(f"{LP_DIR}{m.lower()}_{schema}_lp_training_history.csv")
        if df is None or df.empty:
            continue
        metric_col = 'val_mrr_macro' if 'val_mrr_macro' in df.columns else 'val_mrr'
        sub = df.dropna(subset=[metric_col])
        if sub.empty:
            continue
        _any_lp_history = True
        ax.plot(sub['epoch'], sub[metric_col], marker='o', markersize=3, color=MODEL_COLORS[m], label=m)
    ax.set_title(kg)
    ax.set_xlabel('Epoch')

axes[0].set_ylabel('Val MRR (macro, where logged; else pooled)')
if _any_lp_history:
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.1), ncol=3)
    fig.suptitle('Figure 4c — Link prediction: validation MRR over training, by KG',
                 y=1.18, fontsize=13, fontweight='bold')
    plt.tight_layout()
    save_fig(fig, 'figure4c_lp_training_curves.png')
    plt.show()
else:
    plt.close(fig)
    print("No *_lp_training_history.csv files found yet under link_prediction/.")

[missing] /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/rgcn_base_lp_training_history.csv
[missing] /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/graphsage_base_lp_training_history.csv
[missing] /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/rgcn_sulo_lp_training_history.csv
[missing] /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/link_prediction/graphsage_sulo_lp_training_history.csv
Saved figure: /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure4c_lp_training_curves.png

## 7. Final Results Package Checklist

Cross-checks this notebook's output against `CLAUDE.md`'s Experimental Goals
§6 list. (Table/Figure 4 above is supplementary and intentionally not part of
this checklist — it isn't one of `CLAUDE.md`'s four numbered experiments.)

In [36]:
_checklist = [
    ("Table 1 — Ad-hoc vs SULO KG structural characteristics",
     [f"{GEOM_DIR}rq1a_structural_comparison.csv", f"{ANALYSIS_DIR}table1_kg_structural_comparison.csv"]
     + ([table1d_path] if table1d_path else [])),
    ("Figure 1 — Visual Ad-hoc/SULO subgraph comparison",
     [f"{GEOM_DIR}rq1a_subgraph_comparison.png"]),
    ("Table 2 — GraphSAGE/R-GCN/TransE × Ad-hoc/SULO performance",
     [table2_path]),
    ("Figure 2 — Model performance comparison",
     [f"{ANALYSIS_DIR}figure2a_model_kg_performance.png", f"{ANALYSIS_DIR}figure2b_kg_delta_by_model.png"]),
    ("Table 3 — Feature/relation ablation results",
     [table3_path] if table3_path else []),
    ("Figure 3 — Structural characteristic → performance relationship",
     [f"{ANALYSIS_DIR}figure3_structure_vs_performance.png", f"{ANALYSIS_DIR}figure3b_k_sweep_sensitivity.png"]
     + ([table3b_k_sweep_path] if table3b_k_sweep_path else [])),
]

for name, paths in _checklist:
    ready = any(p and os.path.exists(p) for p in paths)
    print(f"[{'DONE' if ready else 'MISSING'}] {name}")
    for p in paths:
        if p: 
            print(f"    {p}")

[DONE] Table 1 — Ad-hoc vs SULO KG structural characteristics
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/graph_geometry/rq1a_structural_comparison.csv
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/table1_kg_structural_comparison.csv
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/table1d_feature_case_proximity.csv
[DONE] Figure 1 — Visual Ad-hoc/SULO subgraph comparison
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/graph_geometry/rq1a_subgraph_comparison.png
[DONE] Table 2 — GraphSAGE/R-GCN/TransE × Ad-hoc/SULO performance
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/table2_model_kg_performance.csv
[DONE] Figure 2 — Model performance comparison
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure2a_model_kg_performance.png
    /Volumes/westeurope_cac_dwh_prd/ds_ccc/ds-landing/sulo_km/analysis/figure2b_kg_delta_by_model.png
[DONE] Table 3 — Feature/relati